# Practice: deep networks, in NumPy and PyTorch
*AI from scratch, deep networks layer by layer* · lesson: https://ihor.sh/ai/deep-networks.html

Count parameters, watch signals vanish or survive through 20 layers depending on the starting weights, then
train the same spiral network in PyTorch and compare deep against wide at the same size.
Fill in each `...` under `# YOUR CODE HERE`, then run the check under it (✓ = right, ✗ = not yet).

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['For each layer transition from m inputs to n outputs, count weights (m * n) and biases (n).', 'Sum (m * n + n) across all layer pairs zip(sizes, sizes[1:]).'], 2: ['For each scale s, start with activations A = X0.', 'Loop through 20 layers: A = np.tanh(rng.standard_normal((256, 256)) * s @ A).', 'Record final standard deviation A.std() in final_std[name].'], 3: ['Construct deep network: nn.Sequential(nn.Linear(2, 16), nn.Tanh(), nn.Linear(16, 16), nn.Tanh(), nn.Linear(16, 1)).', 'Train using train(deep) and assign result to acc.'], 4: ['Set torch.manual_seed(0).', 'Construct wide network: nn.Sequential(nn.Linear(2, 84), nn.Tanh(), nn.Linear(84, 1)).', 'Train using train(...) and assign result to acc_wide.']}
_ANSWERS = {1: 'cmV0dXJuIHN1bShuICogbSArIG4gZm9yIG0sIG4gaW4gemlwKHNpemVzLCBzaXplc1sxOl0pKQ==', 2: 'ZmluYWxfc3RkID0ge30KZm9yIG5hbWUsIHMgaW4gc2NhbGVzLml0ZW1zKCk6CiAgICBBID0gWDAKICAgIGZvciBfIGluIHJhbmdlKDIwKToKICAgICAgICBBID0gbnAudGFuaChybmcuc3RhbmRhcmRfbm9ybWFsKCgyNTYsIDI1NikpICogcyBAIEEpCiAgICBmaW5hbF9zdGRbbmFtZV0gPSBBLnN0ZCgp', 3: 'ZGVlcCA9IG5uLlNlcXVlbnRpYWwobm4uTGluZWFyKDIsIDE2KSwgbm4uVGFuaCgpLCBubi5MaW5lYXIoMTYsIDE2KSwgbm4uVGFuaCgpLCBubi5MaW5lYXIoMTYsIDEpKQphY2MgPSB0cmFpbihkZWVwKQ==', 4: 'dG9yY2gubWFudWFsX3NlZWQoMCkKYWNjX3dpZGUgPSB0cmFpbihubi5TZXF1ZW50aWFsKG5uLkxpbmVhcigyLCA4NCksIG5uLlRhbmgoKSwgbm4uTGluZWFyKDg0LCAxKSkp'}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn

## 1. Count the knobs
Write `n_params(sizes)`: the total weights and biases of a fully connected network with those layer sizes.

In [ ]:
def n_params(sizes):
    """Compute total number of trainable weights and biases in a fully connected MLP.
    
    Args:
        sizes: List of integers representing layer widths, e.g. [2, 16, 16, 1].
    Returns:
        int: Total number of parameters (weights + biases).
    """
    # YOUR CODE HERE (stuck? hint(1), then answer(1))
    ...

In [ ]:
check(1, lambda: n_params([2, 16, 16, 1]) == 337 and n_params([784, 128, 10]) == 101_770, 
      "A 2-16-16-1 network has 337 parameters; 784-128-10 has 101,770.")
check(1, lambda: n_params([2, 16, 16, 1]) == sum(p.numel() for p in nn.Sequential(nn.Linear(2, 16), nn.Linear(16, 16), nn.Linear(16, 1)).parameters()), 
      "n_params formula must match total parameter count from PyTorch nn.Sequential.")

## 2. How the starting weights decide everything
Push 1000 random inputs through 20 tanh layers of 256 units, with weights drawn at scale `s`. Record the standard
deviation of the activations after the last layer, for s = 0.01, Xavier ($\sqrt{1/256}$) and 0.2.
Set `final_std` as a dict {name: std}.

In [ ]:
rng = np.random.default_rng(0)
X0 = rng.standard_normal((256, 1000))
scales = {"tiny (0.01)": 0.01, "Xavier": np.sqrt(1 / 256), "big (0.2)": 0.2}
final_std = None
# YOUR CODE HERE (stuck? hint(2), then answer(2))
...
print(final_std)

In [ ]:
check(2, lambda: final_std["tiny (0.01)"] < 1e-10, 
      "With tiny initialization scale (0.01), activations should vanish (< 1e-10) after 20 layers.")
check(2, lambda: 0.1 < final_std["Xavier"] < 0.9, 
      "With Xavier initialization, activation variance should remain healthy (std between 0.1 and 0.9).")
check(2, lambda: final_std["big (0.2)"] > 0.8, 
      "With too large initialization (0.2), activations saturate tanh and crowd near ±1 (std > 0.8).")

## 3. The spiral in PyTorch
Build a model using `Tanh` activations: `nn.Sequential(nn.Linear(2, 16), nn.Tanh(), nn.Linear(16, 16), nn.Tanh(), nn.Linear(16, 1))` and train it
with `nn.BCEWithLogitsLoss()` and `torch.optim.SGD(lr=0.5)`, full batch, 4000 steps. Set `acc`.
(PyTorch's `nn.Linear` already starts with sensible random weights.)

In [ ]:
def spiral(m=200, seed=1):
    g = np.random.default_rng(seed)
    t = np.tile(np.linspace(0, 1, m // 2, endpoint=False), 2); c = np.repeat([0, 1], m // 2)
    r, a = 0.15 + 0.85 * t, 3.5 * np.pi * t + c * np.pi + 0.15 * g.standard_normal(m)
    return torch.tensor(np.column_stack([r * np.cos(a), r * np.sin(a)]), dtype=torch.float32), torch.tensor(c, dtype=torch.float32)

Xs, ys = spiral()
def train(model, steps=4000, lr=0.5):
    opt, loss_fn = torch.optim.SGD(model.parameters(), lr=lr), nn.BCEWithLogitsLoss()
    for _ in range(steps):
        opt.zero_grad(); loss_fn(model(Xs).squeeze(1), ys).backward(); opt.step()
    with torch.no_grad():
        return ((model(Xs).squeeze(1) > 0).float() == ys).float().mean().item()

torch.manual_seed(0)
acc = None
# YOUR CODE HERE (stuck? hint(3), then answer(3))
...
print("2-16-16-1:", acc)

In [ ]:
check(3, lambda: acc > 0.95, 
      "The deep 2-16-16-1 architecture should achieve > 95% accuracy on the two-spirals problem.")

## 4. Deep vs wide, same size
A single hidden layer with 84 units also has 337 parameters. Train it the same way (same steps, same learning
rate) and set `acc_wide`. Then, for a fair comparison, try 3 different seeds for each and compare the averages.

In [ ]:
acc_wide = None
# YOUR CODE HERE (stuck? hint(4), then answer(4))
...
print("2-84-1:", acc_wide, " parameters:", n_params([2, 84, 1]))

In [ ]:
check(4, lambda: acc_wide < acc, 
      "At identical parameter budgets (337 parameters), the deep model outperforms the shallow wide model on spirals.")

## 5. What the deep one learned
The decision regions of your deep network over the plane.

In [ ]:
if acc is not None:
    xx, yy = np.meshgrid(np.linspace(-1.2, 1.2, 200), np.linspace(-1.2, 1.2, 200))
    with torch.no_grad():
        zz = deep(torch.tensor(np.column_stack([xx.ravel(), yy.ravel()]), dtype=torch.float32)).reshape(xx.shape).numpy()
    plt.figure(figsize=(5, 5)); plt.contourf(xx, yy, zz > 0, alpha=0.3, cmap="coolwarm")
    plt.scatter(Xs[:, 0], Xs[:, 1], c=ys, cmap="coolwarm", s=12); plt.axis("equal"); plt.show()

## Next
Back to the path: https://ihor.sh/ai/ · stage 3 starts with chapter 11, overfitting. Milestone for stage 2: train a
digit reader from scratch, then in PyTorch (the next notebooks build up to it).